# Selectivity as a Spectrum: A Distribution-Based Framework for Quantifying Multi-Target Activity Patterns Across Protein Families in ChEMBL

**Authors:** Abubakar Siddiq Salihu<sup>a</sup> and Wan Mohd Nuzul Hakimi Wan Salleh<sup>b</sup>

<sup>a</sup> Department of Pure and Industrial Chemistry, Umaru Musa Yar'adua University, PMB 2218, Katsina, Nigeria  
<sup>b</sup> Department of Chemistry, Faculty of Science and Mathematics, Universiti Pendidikan Sultan Idris (UPSI), 35900 Tanjong Malim, Perak, Malaysia

**Corresponding author:** abubakar.salihu@umyu.edu.ng  
**ORCID (A.S. Salihu):** 0000-0002-4425-7524

**Journal:** ChemMedChem (submitted)  
**Data source:** ChEMBL 36 (https://www.ebi.ac.uk/chembl/)

---

## Notebook structure

| Cell | Content |
|------|---------|
| 1 | Imports and configuration |
| 2 | ChEMBL SQL extraction (primary: Ki/Kd) |
| 3 | Deduplication and variance computation |
| 4 | Target family annotation |
| 5 | Compound eligibility filter and scope characterization |
| 6 | Selectivity entropy computation (primary metric) |
| 7 | Cross-family correlation analysis (RQ2) |
| 8 | Save analytical outputs |
| 9 | Publication-quality figures (Figures 1-5) |
| 10 | Manuscript statistics summary |
| 11 | Sensitivity analysis: broader assay set (Ki/Kd/IC50) |

---

In [ ]:
# Cell 1 — Imports and Configuration

import sqlite3
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, ks_2samp, mannwhitneyu
from itertools import combinations
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
# Update CHEMBL_DB to your local ChEMBL 36 SQLite path
CHEMBL_DB  = Path(r'C:\Users\PC\Documents\inha_qsar_project\data\chembl_36\chembl_36_sqlite\chembl_36.db')
OUTPUT_DIR = Path('article_02_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Analysis parameters ────────────────────────────────────────────────────
PCHEMBL_MIN        = 5.0
CONFIDENCE_MIN     = 7
ASSAY_TYPES_PRIMARY = ('Ki', 'Kd')
ASSAY_TYPES_BROAD   = ('Ki', 'Kd', 'IC50')
TOP_K              = 3
N_BOOTSTRAP        = 1000
N_PERMUTATIONS     = 1000
RQ2_MIN_SHARED     = 50
RANDOM_SEED        = 42

np.random.seed(RANDOM_SEED)

# ── Target family keyword map ──────────────────────────────────────────────
FAMILY_KEYWORDS = {
    'Kinase'            : ['kinase', 'growth factor receptor', 'hepatocyte growth factor',
                           'epidermal growth factor receptor', 'fibroblast growth factor receptor',
                           'mast/stem cell growth factor receptor',
                           'vascular endothelial growth factor receptor'],
    'GPCR'              : ['gpcr', 'g protein-coupled', 'g-protein coupled',
                           'dopamine receptor', 'serotonin receptor',
                           '5-hydroxytryptamine receptor', 'adrenergic receptor',
                           'muscarinic acetylcholine receptor', 'opioid receptor',
                           'adenosine receptor', 'cannabinoid receptor',
                           'histamine', 'orexin receptor', 'orexin/hypocretin receptor',
                           'melanocortin receptor', 'nociceptin receptor',
                           'vasopressin', 'corticotropin-releasing',
                           'melanin-concentrating hormone receptor',
                           'prostaglandin', 'chemokine receptor',
                           'angiotensin', 'endothelin receptor',
                           'melatonin receptor', 'substance-p receptor',
                           'substance-k receptor', 'neuromedin-k receptor',
                           'gonadotropin-releasing hormone receptor',
                           'oxytocin receptor', 'thromboxane',
                           'neuropeptide y receptor', 'bradykinin receptor',
                           'growth hormone secretagogue receptor',
                           'calcitonin gene-related peptide',
                           'gastrin/cholecystokinin', 'metabotropic glutamate receptor',
                           'p2y purinoceptor'],
    'Protease'          : ['protease', 'peptidase', 'secretase',
                           'kallikrein', 'coagulation factor', 'prothrombin',
                           'thrombin', 'caspase', 'elastase', 'collagenase',
                           'stromelysin', 'cathepsin', 'matrix metalloproteinase',
                           'furin', 'procathepsin'],
    'Nuclear Receptor'  : ['nuclear receptor', 'androgen receptor',
                           'estrogen receptor', 'glucocorticoid receptor',
                           'progesterone receptor', 'retinoic acid receptor',
                           'peroxisome proliferator', 'aromatase'],
    'Ion Channel'       : ['ion channel', 'potassium channel', 'sodium channel',
                           'calcium channel', 'chloride channel', 'kcnh2',
                           'voltage-gated', 'ligand-gated',
                           'p2x purinoceptor', 'acetylcholine receptor subunit',
                           'gamma-aminobutyric acid receptor'],
    'Epigenetic'        : ['histone', 'bromodomain', 'hdac', 'methyltransferase',
                           'demethylase', 'wd repeat'],
    'Transporter'       : ['transporter', 'slc', 'abc transporter',
                           'serotonin transporter', 'dopamine transporter',
                           'norepinephrine transporter'],
    'Carbonic Anhydrase': ['carbonic anhydrase'],
    'Phosphodiesterase' : ['phosphodiesterase', 'cyclic phosphodiesterase'],
    'Apoptosis'         : ['bcl-2', 'bcl-x', 'mcl-1', 'apoptosis regulator',
                           'e3 ubiquitin-protein ligase mdm2'],
    'Other Enzyme'      : ['lipoxygenase', 'endonuclease', 'poly [adp-ribose]',
                           'sigma non-opioid', 'sigma intracellular',
                           'acetylcholinesterase', 'son of sevenless'],
}

def assign_family(target_name):
    name_lower = str(target_name).lower()
    for family, keywords in FAMILY_KEYWORDS.items():
        if any(kw in name_lower for kw in keywords):
            return family
    return 'Other'

def top_k_mean(values, k=TOP_K):
    arr = sorted(values, reverse=True)
    return np.mean(arr[:k])

print('\u2713 Imports complete')
print(f'\u2713 Output directory: {OUTPUT_DIR}')
print(f'\u2713 Primary assay types: {ASSAY_TYPES_PRIMARY}')
print(f'\u2713 pChEMBL threshold: {PCHEMBL_MIN}')

In [ ]:
# Cell 2 — ChEMBL SQL Extraction (Primary: Ki/Kd, Human targets, Confidence >= 7)

conn = sqlite3.connect(CHEMBL_DB)

query = """
SELECT
    md.chembl_id                        AS compound_chembl_id,
    cs.canonical_smiles                 AS smiles,
    td.chembl_id                        AS target_chembl_id,
    td.pref_name                        AS target_name,
    act.standard_type                   AS assay_type,
    act.pchembl_value                   AS pchembl,
    act.standard_relation               AS relation,
    ass.confidence_score                AS confidence,
    ass.assay_type                      AS assay_category
FROM
    activities          act
    JOIN assays         ass ON act.assay_id       = ass.assay_id
    JOIN target_dictionary td  ON ass.tid         = td.tid
    JOIN molecule_dictionary md ON act.molregno   = md.molregno
    JOIN compound_structures cs ON act.molregno   = cs.molregno
WHERE
    td.organism         = 'Homo sapiens'
    AND td.target_type  = 'SINGLE PROTEIN'
    AND ass.confidence_score >= ?
    AND act.standard_type IN ('Ki', 'Kd')
    AND act.pchembl_value IS NOT NULL
    AND act.pchembl_value >= ?
    AND act.standard_relation = '='
    AND cs.canonical_smiles IS NOT NULL
"""

print('Running extraction query (1-3 minutes)...')
df_raw = pd.read_sql_query(query, conn, params=(CONFIDENCE_MIN, PCHEMBL_MIN))
conn.close()

print(f'\u2713 Raw records extracted : {len(df_raw):,}')
print(f'\u2713 Unique compounds      : {df_raw["compound_chembl_id"].nunique():,}')
print(f'\u2713 Unique targets        : {df_raw["target_chembl_id"].nunique():,}')
print(f'\nAssay type breakdown:')
print(df_raw['assay_type'].value_counts())

In [ ]:
# Cell 3 — Deduplication: median pChEMBL per compound-target pair + variance

df_variance = (
    df_raw
    .groupby(['compound_chembl_id', 'target_chembl_id'])
    .agg(
        pchembl_median   = ('pchembl', 'median'),
        pchembl_mean     = ('pchembl', 'mean'),
        pchembl_variance = ('pchembl', 'var'),
        assay_count      = ('pchembl', 'count'),
        target_name      = ('target_name', 'first'),
        smiles           = ('smiles', 'first'),
        assay_category   = ('assay_category', 'first'),
        assay_type       = ('assay_type', 'first'),
    )
    .reset_index()
)
df_variance['pchembl_variance'] = df_variance['pchembl_variance'].fillna(0)
df_clean = df_variance.copy()

print(f'\u2713 Unique compound-target pairs : {len(df_clean):,}')
print(f'\u2713 Unique compounds             : {df_clean["compound_chembl_id"].nunique():,}')
print(f'\u2713 Unique targets               : {df_clean["target_chembl_id"].nunique():,}')
print(f'\nPairs with >1 assay (variance computable): '
      f'{(df_clean["assay_count"] > 1).sum():,} '
      f'({(df_clean["assay_count"] > 1).mean()*100:.1f}%)')

In [ ]:
# Cell 4 — Target Family Annotation

df_clean['target_family'] = df_clean['target_name'].apply(assign_family)

family_counts        = df_clean['target_family'].value_counts()
target_family_counts = df_clean.groupby('target_family')['target_chembl_id'].nunique().sort_values(ascending=False)
compound_fam_counts  = df_clean.groupby('target_family')['compound_chembl_id'].nunique().sort_values(ascending=False)

print('Target family annotation summary:')
print(f'{"Family":<22} {"Pairs":>8} {"Targets":>9} {"Compounds":>11}')
print('-' * 54)
for fam in family_counts.index:
    print(f'{fam:<22} {family_counts[fam]:>8,} '
          f'{target_family_counts.get(fam, 0):>9,} '
          f'{compound_fam_counts.get(fam, 0):>11,}')

print(f'\n\u2713 Classified  : {(df_clean["target_family"] != "Other").mean()*100:.1f}%')
print(f'\u2713 Unclassified: {(df_clean["target_family"] == "Other").mean()*100:.1f}%')

In [ ]:
# Cell 5 — Compound Eligibility Filter and Scope Characterization

compound_families = (
    df_clean[df_clean['target_family'] != 'Other']
    .groupby('compound_chembl_id')['target_family']
    .nunique().reset_index()
    .rename(columns={'target_family': 'n_families'})
)
compound_targets = (
    df_clean[df_clean['target_family'] != 'Other']
    .groupby('compound_chembl_id')['target_chembl_id']
    .nunique().reset_index()
    .rename(columns={'target_chembl_id': 'n_targets'})
)
compound_scope = compound_families.merge(compound_targets, on='compound_chembl_id')

total          = df_clean['compound_chembl_id'].nunique()
single_target  = (compound_scope['n_targets'] == 1).sum()
multi_same_fam = ((compound_scope['n_targets'] > 1) & (compound_scope['n_families'] == 1)).sum()
multi_family   = (compound_scope['n_families'] >= 2).sum()
unclassified   = total - len(compound_scope)

print('\u2500\u2500 Scope Breakdown (Figure 1A data) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'Total compounds (Ki/Kd subset)     : {total:>8,}')
print(f'  Single-target                    : {single_target:>8,} ({single_target/total*100:.1f}%)')
print(f'  Multi-target, same family        : {multi_same_fam:>8,} ({multi_same_fam/total*100:.1f}%)')
print(f'  Multi-family active (study set)  : {multi_family:>8,} ({multi_family/total*100:.1f}%)')
print(f'  Only Other-family targets        : {unclassified:>8,} ({unclassified/total*100:.1f}%)')

multifam_ids = compound_scope[compound_scope['n_families'] >= 2]['compound_chembl_id'].tolist()
df_multifam  = df_clean[
    (df_clean['compound_chembl_id'].isin(multifam_ids)) &
    (df_clean['target_family'] != 'Other')
].copy()

print(f'\n\u2500\u2500 Analysis Subset \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'\u2713 Multi-family compounds  : {df_multifam["compound_chembl_id"].nunique():>8,}')
print(f'\u2713 Compound-target pairs   : {len(df_multifam):>8,}')
print(f'\u2713 Unique targets          : {df_multifam["target_chembl_id"].nunique():>8,}')
print(f'\nFamily distribution in analysis subset:')
print(df_multifam['target_family'].value_counts())

In [ ]:
# Cell 6 — Selectivity Entropy Computation
#
# Primary metric: normalized magnitude-aware Shannon entropy
#   w(i)     = 10^pChEMBL(i)          [inverse-concentration weighting, Eq. 1]
#   p(i)     = w(i) / sum_j w(j)       [probability vector, Eq. 2]
#   H        = -sum p(i) log2 p(i)     [Shannon entropy, Eq. 3]
#   H_norm   = H / log2(n)             [normalized entropy in [0,1], Eq. 4]

records = []

for cmpd_id, group in df_multifam.groupby('compound_chembl_id'):
    pchembl_vals = group['pchembl_median'].values
    n_targets    = len(pchembl_vals)
    n_families   = group['target_family'].nunique()

    # Magnitude-aware entropy (primary)
    weights      = 10 ** pchembl_vals
    probs        = weights / weights.sum()
    h_raw        = -np.sum(probs * np.log2(probs + 1e-12))
    h_norm       = h_raw / np.log2(n_targets) if n_targets > 1 else 0.0

    # Potency metrics
    potency_topk = top_k_mean(pchembl_vals, k=TOP_K)
    max_pchembl  = pchembl_vals.max()

    records.append({
        'compound_chembl_id'     : cmpd_id,
        'smiles'                 : group['smiles'].iloc[0],
        'n_targets'              : n_targets,
        'n_families'             : n_families,
        'selectivity_entropy'    : h_norm,
        'selectivity_entropy_raw': h_raw,
        'potency_topk'           : potency_topk,
        'max_pchembl'            : max_pchembl,
        'families'               : '|'.join(sorted(set(group['target_family'].values))),
    })

df_metrics = pd.DataFrame(records)

print(f'\u2713 Compounds with metrics: {len(df_metrics):,}')
print(f'\nSelectivity entropy (primary):')
print(df_metrics['selectivity_entropy'].describe().round(3))
print(f'\nCorrelation entropy vs top-k potency : {df_metrics["selectivity_entropy"].corr(df_metrics["potency_topk"]):.3f}')
print(f'Correlation entropy vs max pChEMBL   : {df_metrics["selectivity_entropy"].corr(df_metrics["max_pchembl"]):.3f}')
print(f'Correlation entropy vs n_targets     : {df_metrics["selectivity_entropy"].corr(df_metrics["n_targets"]):.3f}')

In [ ]:
# Cell 7 — Cross-Family Correlation Analysis (RQ2)

# ── Within-family entropy distributions ───────────────────────────────────
family_pair_counts = df_multifam['target_family'].value_counts()
rq2_families       = family_pair_counts[family_pair_counts >= RQ2_MIN_SHARED].index.tolist()

print(f'Families meeting >= {RQ2_MIN_SHARED} pairs threshold: {rq2_families}')
print(f'\nWithin-family entropy (mean ± std):')
print(f'{"Family":<22} {"n":>6}  {"mean":>6}  {"std":>6}  {"median":>7}')
print('-' * 55)
for fam in sorted(rq2_families):
    sub = df_metrics[df_metrics['families'].str.contains(fam)]['selectivity_entropy']
    print(f'{fam:<22} {len(sub):>6,}  {sub.mean():>6.3f}  {sub.std():>6.3f}  {sub.median():>7.3f}')

# ── Inter-family Spearman correlations (bootstrapped) ─────────────────────
corr_results = []
for fam_a, fam_b in combinations(rq2_families, 2):
    mask   = (df_metrics['families'].str.contains(fam_a) &
              df_metrics['families'].str.contains(fam_b))
    shared = df_metrics[mask]
    if len(shared) < 10:
        continue
    r_obs, p_obs = spearmanr(shared['selectivity_entropy'], shared['n_targets'])
    boot_r = [spearmanr(
                shared.sample(n=len(shared), replace=True)['selectivity_entropy'],
                shared.sample(n=len(shared), replace=True)['n_targets']
               )[0] for _ in range(N_BOOTSTRAP)]
    corr_results.append({
        'family_a' : fam_a, 'family_b' : fam_b,
        'n_shared' : len(shared),
        'r_obs'    : r_obs,   'p_obs'    : p_obs,
        'ci_low'   : np.percentile(boot_r, 2.5),
        'ci_high'  : np.percentile(boot_r, 97.5),
    })

df_corr         = pd.DataFrame(corr_results)
df_corr_primary = df_corr[df_corr['n_shared'] >= RQ2_MIN_SHARED].copy()

print(f'\nPrimary cross-family pairs (n >= {RQ2_MIN_SHARED}):')
print(f'{"Pair":<35} {"n":>5}  {"r":>7}  {"p":>12}  {"sig":>5}')
print('-' * 68)
for _, row in df_corr_primary.sort_values('p_obs').iterrows():
    pair = f"{row['family_a']} vs {row['family_b']}"
    sig  = ('***' if row['p_obs'] < 0.001 else
            '**'  if row['p_obs'] < 0.01  else
            '*'   if row['p_obs'] < 0.05  else 'ns')
    print(f'{pair:<35} {row["n_shared"]:>5,}  {row["r_obs"]:>7.3f}  {row["p_obs"]:>12.2e}  {sig:>5}')

# ── Partial correlations (confound control) ────────────────────────────────
def partial_corr(df, x, y, z):
    """Partial Spearman correlation of x and y controlling for z."""
    from scipy.stats import spearmanr
    r_xy = spearmanr(df[x], df[y])[0]
    r_xz = spearmanr(df[x], df[z])[0]
    r_yz = spearmanr(df[y], df[z])[0]
    denom = np.sqrt((1 - r_xz**2) * (1 - r_yz**2))
    if denom == 0:
        return np.nan, np.nan
    r_p = (r_xy - r_xz * r_yz) / denom
    n   = len(df)
    t   = r_p * np.sqrt(n - 2) / np.sqrt(1 - r_p**2 + 1e-12)
    from scipy.stats import t as t_dist
    p   = 2 * t_dist.sf(abs(t), df=n-2)
    return r_p, p

print(f'\nPartial correlation analysis for primary significant pairs:')
for fam_a, fam_b in [('Transporter', 'Ion Channel'), ('GPCR', 'Kinase')]:
    mask   = (df_metrics['families'].str.contains(fam_a) &
              df_metrics['families'].str.contains(fam_b))
    shared = df_metrics[mask].copy()
    r_raw, p_raw   = spearmanr(shared['selectivity_entropy'], shared['n_targets'])
    r_pc1, p_pc1   = partial_corr(shared, 'selectivity_entropy', 'n_targets', 'max_pchembl')
    r_pc2, p_pc2   = partial_corr(shared, 'selectivity_entropy', 'n_targets', 'n_targets')
    print(f'\n{fam_a} vs {fam_b} (n={len(shared)}):')
    print(f'  Raw r                           : {r_raw:.4f}  p={p_raw:.4e}')
    print(f'  Partial r (ctrl max_pchembl)    : {r_pc1:.4f}  p={p_pc1:.4e}')

# ── GPCR-removed sensitivity check ────────────────────────────────────────
no_gpcr = df_metrics[~df_metrics['families'].str.contains('GPCR')]
print(f'\nGPCR-removed sensitivity:')
print(f'  Compounds without GPCR : {len(no_gpcr):,}')
print(f'  Mean entropy (full)    : {df_metrics["selectivity_entropy"].mean():.4f}')
print(f'  Mean entropy (no GPCR) : {no_gpcr["selectivity_entropy"].mean():.4f}')

In [ ]:
# Cell 8 — Save Analytical Outputs

df_metrics.to_parquet(OUTPUT_DIR / 'df_metrics.parquet', index=False)
df_multifam.to_parquet(OUTPUT_DIR / 'df_multifam.parquet', index=False)
df_variance.to_parquet(OUTPUT_DIR / 'df_variance.parquet', index=False)
df_corr.to_parquet(OUTPUT_DIR / 'df_corr_all.parquet', index=False)
df_corr_primary.to_parquet(OUTPUT_DIR / 'df_corr_primary.parquet', index=False)

scope_data = {
    'total_compounds'  : 183150,
    'single_target'    : 99576,
    'multi_same_family': 55861,
    'multi_family'     : 3649,
    'other_only'       : 24064,
}
with open(OUTPUT_DIR / 'scope_data.pkl', 'wb') as f:
    pickle.dump(scope_data, f)

print('\u2713 Saved:')
for f in sorted(OUTPUT_DIR.glob('*')):
    print(f'  {f.name}')

In [ ]:
# Cell 9 — Publication-Quality Figures (300 DPI, PNG + SVG)

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

FAMILY_COLORS = {
    'GPCR': '#4C72B0', 'Kinase': '#DD8452', 'Transporter': '#55A868',
    'Protease': '#C44E52', 'Ion Channel': '#8172B3',
    'Carbonic Anhydrase': '#937860', 'Other Enzyme': '#DA8BC3',
    'Epigenetic': '#8C8C8C', 'Nuclear Receptor': '#CCB974',
    'Phosphodiesterase': '#64B5CD', 'Apoptosis': '#E377C2',
}

# ── Figure 1 — Dataset Overview ────────────────────────────────────────────
fig1, axes = plt.subplots(1, 3, figsize=(14, 5))
fig1.suptitle('Figure 1. Dataset Overview and Scope Definition', fontsize=13, fontweight='bold', y=1.02)

ax = axes[0]
labels = ['Single-target\n(54.4%)', 'Multi-target\nsame family\n(30.5%)',
          'Multi-family\nactive\n(2.0%)', 'Unclassified\ntargets\n(13.1%)']
sizes  = [99576, 55861, 3649, 24064]
colors = ['#AEC6E8', '#FFBB78', '#4C72B0', '#D3D3D3']
wedges, _ = ax.pie(sizes, colors=colors, startangle=90,
                   wedgeprops=dict(edgecolor='white', linewidth=1.5))
ax.legend(wedges, labels, loc='lower center', bbox_to_anchor=(0.5, -0.22), fontsize=8, frameon=False)
ax.set_title('A. Compound Scope\n(n = 183,150 total)', fontweight='bold')

ax = axes[1]
fam_order  = df_multifam['target_family'].value_counts().index.tolist()
fam_counts = df_multifam['target_family'].value_counts()
bar_colors = [FAMILY_COLORS.get(f, '#999999') for f in fam_order]
bars = ax.barh(fam_order, fam_counts.values, color=bar_colors, edgecolor='white')
ax.set_xlabel('Compound-target pairs')
ax.set_title('B. Family Distribution\n(multi-family subset, n = 3,649)', fontweight='bold')
for bar, val in zip(bars, fam_counts.values):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=8)
ax.set_xlim(0, fam_counts.max() * 1.15)

ax = axes[2]
families_to_plot = [f for f in fam_order if f in FAMILY_COLORS][:8]
data_to_plot = [df_multifam[df_multifam['target_family'] == f]['pchembl_median'].values for f in families_to_plot]
bp = ax.boxplot(data_to_plot, vert=False, patch_artist=True,
                medianprops=dict(color='white', linewidth=2),
                flierprops=dict(marker='o', markersize=2, alpha=0.3))
for patch, fam in zip(bp['boxes'], families_to_plot):
    patch.set_facecolor(FAMILY_COLORS.get(fam, '#999999'))
    patch.set_alpha(0.8)
ax.set_yticklabels(families_to_plot, fontsize=8)
ax.set_xlabel('pChEMBL value (median per pair)')
ax.set_title('C. Activity Distribution\nby Target Family', fontweight='bold')
ax.axvline(x=7, color='grey', linestyle='--', alpha=0.5, linewidth=1)
plt.tight_layout()
fig1.savefig(OUTPUT_DIR / 'Figure1_dataset_overview.png')
fig1.savefig(OUTPUT_DIR / 'Figure1_dataset_overview.svg')
plt.show()
print('\u2713 Figure 1 saved')

# ── Figure 2 — Selectivity Spectrum ───────────────────────────────────────
fig2, axes = plt.subplots(1, 2, figsize=(13, 5))
fig2.suptitle('Figure 2. The Selectivity Spectrum of Multi-Family Active Compounds', fontsize=13, fontweight='bold', y=1.02)

ax = axes[0]
ax.hist(df_metrics['selectivity_entropy'], bins=50, color='#4C72B0', edgecolor='white', alpha=0.85)
ax.axvline(df_metrics['selectivity_entropy'].median(), color='#C44E52', linestyle='--', linewidth=1.5,
           label=f"Median = {df_metrics['selectivity_entropy'].median():.3f}")
ax.axvline(df_metrics['selectivity_entropy'].mean(), color='#DD8452', linestyle=':', linewidth=1.5,
           label=f"Mean = {df_metrics['selectivity_entropy'].mean():.3f}")
ax.set_xlabel('Normalized selectivity entropy')
ax.set_ylabel('Number of compounds')
ax.set_title('A. Selectivity Entropy Distribution\n(n = 3,649 compounds)', fontweight='bold')
ax.legend(frameon=False, fontsize=9)
ax.text(0.02, 0.92, 'Selective\n(low entropy)', transform=ax.transAxes, fontsize=8, color='#555555', va='top')
ax.text(0.75, 0.92, 'Promiscuous\n(high entropy)', transform=ax.transAxes, fontsize=8, color='#555555', va='top')

ax = axes[1]
primary_fam = df_metrics['families'].apply(lambda x: x.split('|')[0])
for fam in primary_fam.unique():
    mask = primary_fam == fam
    ax.scatter(df_metrics.loc[mask, 'selectivity_entropy'], df_metrics.loc[mask, 'potency_topk'],
               c=FAMILY_COLORS.get(fam, '#999999'), alpha=0.35, s=12, label=fam)
ax.set_xlabel('Normalized selectivity entropy')
ax.set_ylabel('Top-3 mean pChEMBL (potency)')
ax.set_title('B. 2D Selectivity-Potency Space\n(colored by primary family)', fontweight='bold')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7, frameon=False, markerscale=2)
ax.axhline(y=7, color='grey', linestyle='--', alpha=0.4, linewidth=1)
ax.axvline(x=0.5, color='grey', linestyle='--', alpha=0.4, linewidth=1)
for txt, pos in [('Selective\nhigh-potency', (0.02, 0.97)), ('Promiscuous\nhigh-potency', (0.65, 0.97)),
                  ('Selective\nlow-potency', (0.02, 0.18)), ('Promiscuous\nlow-potency', (0.65, 0.18))]:
    ax.text(pos[0], pos[1], txt, transform=ax.transAxes, fontsize=7.5, color='#333333', va='top', alpha=0.7)
plt.tight_layout()
fig2.savefig(OUTPUT_DIR / 'Figure2_selectivity_spectrum.png')
fig2.savefig(OUTPUT_DIR / 'Figure2_selectivity_spectrum.svg')
plt.show()
print('\u2713 Figure 2 saved')

# ── Figure 3 — Cross-Family Correlation Heatmap ───────────────────────────
fig3, axes = plt.subplots(1, 2, figsize=(14, 6))
fig3.suptitle('Figure 3. Inter-Family Selectivity Entropy Correlations', fontsize=13, fontweight='bold', y=1.02)

all_fams     = sorted(rq2_families)
corr_matrix  = pd.DataFrame(np.nan, index=all_fams, columns=all_fams)
corr_primary = pd.DataFrame(np.nan, index=all_fams, columns=all_fams)
np.fill_diagonal(corr_matrix.values, 1.0)
np.fill_diagonal(corr_primary.values, 1.0)

for _, row in df_corr.iterrows():
    fa, fb = row['family_a'], row['family_b']
    if fa in all_fams and fb in all_fams:
        corr_matrix.loc[fa, fb] = corr_matrix.loc[fb, fa] = row['r_obs']
for _, row in df_corr_primary.iterrows():
    fa, fb = row['family_a'], row['family_b']
    if fa in all_fams and fb in all_fams:
        corr_primary.loc[fa, fb] = corr_primary.loc[fb, fa] = row['r_obs']

norm = TwoSlopeNorm(vmin=-0.7, vcenter=0, vmax=0.7)
for ax, mat, title in zip(axes,
    [corr_matrix, corr_primary],
    ['A. All Family Pairs\n(Spearman r, NaN = n<10 shared)',
     'B. Primary Pairs (n\u226550 shared)\n(* p<0.05  ** p<0.01  *** p<0.001)']):
    sns.heatmap(mat.astype(float), ax=ax, mask=mat.isna(), cmap='RdBu_r', norm=norm, center=0,
                annot=True, fmt='.2f', annot_kws={'size': 8}, linewidths=0.5, linecolor='white',
                cbar_kws={'label': 'Spearman r', 'shrink': 0.8})
    ax.set_title(title, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

for _, row in df_corr_primary.iterrows():
    fa, fb = row['family_a'], row['family_b']
    if fa in all_fams and fb in all_fams:
        xi = all_fams.index(fb); yi = all_fams.index(fa)
        star = ('***' if row['p_obs'] < 0.001 else '**' if row['p_obs'] < 0.01
                else '*' if row['p_obs'] < 0.05 else '')
        if star:
            axes[1].text(xi + 0.5, yi + 0.75, star, ha='center', fontsize=7, color='black', fontweight='bold')

plt.tight_layout()
fig3.savefig(OUTPUT_DIR / 'Figure3_crossfamily_correlations.png')
fig3.savefig(OUTPUT_DIR / 'Figure3_crossfamily_correlations.svg')
plt.show()
print('\u2713 Figure 3 saved')

# ── Figure 4 — Signal Reliability ─────────────────────────────────────────
fig4, axes = plt.subplots(1, 3, figsize=(14, 5))
fig4.suptitle('Figure 4. Signal Reliability Assessment', fontsize=13, fontweight='bold', y=1.02)

df_rq1 = df_metrics.merge(
    df_variance.groupby('compound_chembl_id').agg(
        mean_variance    = ('pchembl_variance', 'mean'),
        pct_single_assay = ('assay_count', lambda x: (x == 1).mean())
    ).reset_index(), on='compound_chembl_id', how='left')

ax = axes[0]
ax.scatter(df_rq1['mean_variance'], df_rq1['selectivity_entropy'], alpha=0.2, s=8, color='#4C72B0')
r_val, p_val = spearmanr(df_rq1['mean_variance'].fillna(0), df_rq1['selectivity_entropy'])
ax.set_xlabel('Mean pChEMBL variance across assays')
ax.set_ylabel('Selectivity entropy')
ax.set_title(f'A. Entropy vs Assay Variance\n(r={r_val:.3f}, p={p_val:.3f})', fontweight='bold')
ax.set_xlim(-0.1, 3)

ax = axes[1]
bins_b   = [0, 0.25, 0.5, 0.75, 1.01]
labels_b = ['0-25%', '25-50%', '50-75%', '75-100%']
df_rq1['single_assay_bin'] = pd.cut(df_rq1['pct_single_assay'], bins=bins_b, labels=labels_b)
grp = df_rq1.groupby('single_assay_bin', observed=True)['selectivity_entropy']
ax.bar(grp.mean().index, grp.mean().values, yerr=grp.sem().values,
       color='#DD8452', edgecolor='white', capsize=4, alpha=0.85)
ax.set_xlabel('Proportion of single-assay measurements')
ax.set_ylabel('Mean selectivity entropy')
ax.set_title('B. Entropy by Single-Assay Proportion\n(reliability check)', fontweight='bold')

ax = axes[2]
single = df_rq1[df_rq1['pct_single_assay'] == 1.0]['selectivity_entropy']
multi  = df_rq1[df_rq1['pct_single_assay'] <  1.0]['selectivity_entropy']
ax.hist(single, bins=40, alpha=0.6, color='#C44E52', density=True, label=f'All single-assay (n={len(single):,})')
ax.hist(multi,  bins=40, alpha=0.6, color='#55A868', density=True, label=f'>=1 multi-assay (n={len(multi):,})')
ax.set_xlabel('Selectivity entropy')
ax.set_ylabel('Density')
ax.set_title('C. Single vs Multi-Assay\nEntropy Distributions', fontweight='bold')
ax.legend(frameon=False, fontsize=9)

plt.tight_layout()
fig4.savefig(OUTPUT_DIR / 'Figure4_signal_reliability.png')
fig4.savefig(OUTPUT_DIR / 'Figure4_signal_reliability.svg')
plt.show()
print('\u2713 Figure 4 saved')

# ── Figure 5 — Case Studies ────────────────────────────────────────────────
fig5, axes = plt.subplots(1, 3, figsize=(14, 5))
fig5.suptitle('Figure 5. Exemplar Compounds Across the Selectivity Spectrum', fontsize=13, fontweight='bold', y=1.02)

df_sorted = df_metrics.sort_values('selectivity_entropy')
low_ex  = df_sorted.iloc[10]
mid_ex  = df_sorted.iloc[len(df_sorted) // 2]
high_ex = df_sorted.iloc[-10]

for ax, exemplar, label, color in zip(
    axes,
    [low_ex, mid_ex, high_ex],
    ['Low entropy\n(selective)', 'Mid entropy\n(intermediate)', 'High entropy\n(promiscuous)'],
    ['#55A868', '#DD8452', '#C44E52']
):
    cmpd_id   = exemplar['compound_chembl_id']
    cmpd_data = df_multifam[df_multifam['compound_chembl_id'] == cmpd_id]
    fams      = cmpd_data['target_family'].values
    pvals     = cmpd_data['pchembl_median'].values
    sort_idx  = np.argsort(pvals)[::-1]
    fams_s    = fams[sort_idx][:10]
    pvals_s   = pvals[sort_idx][:10]
    ax.barh(range(len(fams_s)), pvals_s,
            color=[FAMILY_COLORS.get(f, '#999999') for f in fams_s],
            edgecolor='white', alpha=0.85)
    ax.set_yticks(range(len(fams_s)))
    ax.set_yticklabels([f[:15] for f in fams_s], fontsize=8)
    ax.set_xlabel('pChEMBL value')
    ax.set_xlim(4, 11)
    ax.axvline(x=7, color='grey', linestyle='--', alpha=0.5, linewidth=1)
    ax.set_title(f'{label}\n{cmpd_id}\nH={exemplar["selectivity_entropy"]:.3f}, n={exemplar["n_targets"]} targets',
                 fontweight='bold', color=color, fontsize=9)

plt.tight_layout()
fig5.savefig(OUTPUT_DIR / 'Figure5_case_studies.png')
fig5.savefig(OUTPUT_DIR / 'Figure5_case_studies.svg')
plt.show()
print('\u2713 Figure 5 saved')
print(f'\n\u2713 All 5 figures saved to: {OUTPUT_DIR}')

In [ ]:
# Cell 10 — Manuscript Statistics Summary

print('=' * 65)
print('ARTICLE 02 \u2014 MANUSCRIPT STATISTICS SUMMARY')
print('=' * 65)

print('\n\u2500\u2500 1. DATASET \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'  ChEMBL version              : 36')
print(f'  Unique compound-target pairs: {len(df_clean):,}')
print(f'  Unique compounds            : {df_clean["compound_chembl_id"].nunique():,}')
print(f'  Unique targets              : {df_clean["target_chembl_id"].nunique():,}')

print('\n\u2500\u2500 2. SCOPE \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'  Multi-family compounds      : {df_multifam["compound_chembl_id"].nunique():,} (2.0% of total)')

se = df_metrics['selectivity_entropy']
print('\n\u2500\u2500 3. SELECTIVITY ENTROPY \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'  Mean \u00b1 SD         : {se.mean():.3f} \u00b1 {se.std():.3f}')
print(f'  Median [IQR]       : {se.median():.3f} [{se.quantile(0.25):.3f}\u2013{se.quantile(0.75):.3f}]')
print(f'  Skewness           : {se.skew():.3f}')

r_ep, p_ep = spearmanr(df_metrics['selectivity_entropy'], df_metrics['potency_topk'])
r_em, p_em = spearmanr(df_metrics['selectivity_entropy'], df_metrics['max_pchembl'])
print('\n\u2500\u2500 4. ENTROPY-POTENCY RELATIONSHIP \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'  Entropy vs top-3 mean pChEMBL: r={r_ep:.3f}, p={p_ep:.4f}')
print(f'  Entropy vs max pChEMBL       : r={r_em:.3f}, p={p_em:.4f}')

print('\n\u2500\u2500 5. CROSS-FAMILY CORRELATIONS \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'  Primary pairs (n>=50): {len(df_corr_primary)}')
for _, row in df_corr_primary.sort_values('p_obs').iterrows():
    sig = ('***' if row['p_obs'] < 0.001 else '**' if row['p_obs'] < 0.01
           else '*' if row['p_obs'] < 0.05 else 'ns')
    print(f'  {row["family_a"]:12} vs {row["family_b"]:16}: r={row["r_obs"]:+.3f}  p={row["p_obs"]:.2e}  {sig}')

print('\n' + '=' * 65)
print('END OF SUMMARY')
print('=' * 65)

In [ ]:
# Cell 11 — Sensitivity Analysis: Broader Assay Set (Ki, Kd, IC50)

print('Running sensitivity analysis with broader assay set (Ki, Kd, IC50)...')

conn = sqlite3.connect(CHEMBL_DB)
query_broad = """
SELECT
    md.chembl_id        AS compound_chembl_id,
    cs.canonical_smiles AS smiles,
    td.chembl_id        AS target_chembl_id,
    td.pref_name        AS target_name,
    act.standard_type   AS assay_type,
    act.pchembl_value   AS pchembl
FROM activities act
JOIN assays ass ON act.assay_id = ass.assay_id
JOIN target_dictionary td ON ass.tid = td.tid
JOIN molecule_dictionary md ON act.molregno = md.molregno
JOIN compound_structures cs ON act.molregno = cs.molregno
WHERE td.organism = 'Homo sapiens'
  AND td.target_type = 'SINGLE PROTEIN'
  AND ass.confidence_score >= ?
  AND act.standard_type IN ('Ki', 'Kd', 'IC50')
  AND act.pchembl_value IS NOT NULL
  AND act.pchembl_value >= ?
  AND act.standard_relation = '='
  AND cs.canonical_smiles IS NOT NULL
"""
df_broad_raw = pd.read_sql_query(query_broad, conn, params=(CONFIDENCE_MIN, PCHEMBL_MIN))
conn.close()

df_broad_clean = (
    df_broad_raw
    .groupby(['compound_chembl_id', 'target_chembl_id'])
    .agg(pchembl_median=('pchembl', 'median'), target_name=('target_name', 'first'),
         smiles=('smiles', 'first'))
    .reset_index()
)
df_broad_clean['target_family'] = df_broad_clean['target_name'].apply(assign_family)

compound_fam_broad = (
    df_broad_clean[df_broad_clean['target_family'] != 'Other']
    .groupby('compound_chembl_id')['target_family'].nunique()
    .reset_index().rename(columns={'target_family': 'n_families'})
)
multifam_broad_ids = compound_fam_broad[compound_fam_broad['n_families'] >= 2]['compound_chembl_id'].tolist()
df_broad_multifam  = df_broad_clean[
    (df_broad_clean['compound_chembl_id'].isin(multifam_broad_ids)) &
    (df_broad_clean['target_family'] != 'Other')
].copy()

broad_records = []
for cmpd_id, group in df_broad_multifam.groupby('compound_chembl_id'):
    vals    = group['pchembl_median'].values
    n       = len(vals)
    weights = 10 ** vals
    probs   = weights / weights.sum()
    h       = -np.sum(probs * np.log2(probs + 1e-12))
    h_norm  = h / np.log2(n) if n > 1 else 0.0
    broad_records.append({'compound_chembl_id': cmpd_id,
                          'selectivity_entropy': h_norm,
                          'n_targets': n,
                          'max_pchembl': vals.max(),
                          'families': '|'.join(sorted(set(group['target_family'].values)))})
df_broad_metrics = pd.DataFrame(broad_records)

se_p = df_metrics['selectivity_entropy']
se_b = df_broad_metrics['selectivity_entropy']
ks_stat, ks_p = ks_2samp(se_p, se_b)

print(f'\n{"":30} {"Primary (Ki/Kd)":>18} {"Broad (Ki/Kd/IC50)":>20}')
print(f'  n compounds           : {len(se_p):>18,} {len(se_b):>20,}')
print(f'  Mean \u00b1 SD             : {se_p.mean():.3f} \u00b1 {se_p.std():.3f}   {se_b.mean():.3f} \u00b1 {se_b.std():.3f}')
print(f'  KS test               : stat={ks_stat:.4f}, p={ks_p:.4f}')

broad_corr_records = []
for fam_a, fam_b in combinations(rq2_families, 2):
    mask   = (df_broad_metrics['families'].str.contains(fam_a) &
              df_broad_metrics['families'].str.contains(fam_b))
    shared = df_broad_metrics[mask]
    if len(shared) < RQ2_MIN_SHARED:
        continue
    r_obs, p_obs = spearmanr(shared['selectivity_entropy'], shared['n_targets'])
    broad_corr_records.append({'family_a': fam_a, 'family_b': fam_b,
                                'n_shared': len(shared), 'r_broad': r_obs, 'p_broad': p_obs})
df_broad_corr = pd.DataFrame(broad_corr_records)

df_compare = df_corr_primary[['family_a','family_b','r_obs','p_obs']].merge(
    df_broad_corr[['family_a','family_b','r_broad','p_broad']], on=['family_a','family_b'], how='outer')

print(f'\nCross-family correlation comparison:')
print(f'  {"Pair":<35} {"r_primary":>10} {"r_broad":>9}')
print('  ' + '-' * 57)
for _, row in df_compare.sort_values('r_obs').iterrows():
    pair  = f"{row['family_a']} vs {row['family_b']}"
    r_b   = f"{row['r_broad']:.3f}" if pd.notna(row['r_broad']) else 'n/a'
    r_p   = f"{row['r_obs']:.3f}"   if pd.notna(row['r_obs'])   else 'n/a'
    print(f'  {pair:<35} {r_p:>10} {r_b:>9}')

df_broad_metrics.to_parquet(OUTPUT_DIR / 'df_broad_metrics.parquet', index=False)
print(f'\n\u2713 Sensitivity analysis complete. Results saved to: {OUTPUT_DIR}')